# Global.health Line-list Data

This notebook demonstrates how to access standardized, curated line-list data for pandemics and epidemics from [Global.health](https://global.health/).

## Data Source
- **Provider**: Global.health
- **Coverage**: Global
- **License**: CC BY 4.0
- **Update Frequency**: Daily (during active outbreaks)

## Available Datasets
- COVID-19 line-list data
- Monkeypox line-list data
- Other outbreak data (as available)

In [1]:
import sys
sys.path.insert(0, '../../')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from epidatasets.sources.global_health import GlobalHealthAccessor

# Set plot style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Initialize accessor
gh = GlobalHealthAccessor()

## 1. List Available Datasets

In [2]:
# Display available datasets
datasets = gh.list_available_datasets()
datasets

,dataset_name,description,url,available
0,COVID-19,Global COVID-19 line-list data,https://raw.githubusercontent.com/globaldothea...,True
1,Monkeypox,Global Monkeypox line-list data,https://raw.githubusercontent.com/globaldothea...,True


## 2. Get Outbreak Metadata

In [3]:
# Get COVID-19 outbreak metadata
covid_metadata = gh.get_outbreak_metadata("COVID-19")

print("COVID-19 Outbreak Metadata:")
print("="*50)
for key, value in covid_metadata.items():
    if key == "countries" and value:
        print(f"{key}: {len(value)} countries")
        print(f"  Top 10: {', '.join(value[:10])}")
    else:
        print(f"{key}: {value}")

COVID-19 Outbreak Metadata:
disease: COVID-19
description: Global COVID-19 line-list data
url: https://raw.githubusercontent.com/globaldothealth/list/main/data/cases.csv
error: Failed to download data from https://raw.githubusercontent.com/globaldothealth/list/main/data/cases.csv: HTTP Error 404: Not Found


## 3. Load Case Data

In [4]:
# Get COVID-19 case data for Brazil
# Note: Loading full dataset may take time
print("Loading COVID-19 data for Brazil...")
brazil_covid = gh.get_case_data(
    disease="COVID-19",
    country="Brazil",
    date_range=("2022-01-01", "2022-12-31")
)

print(f"Loaded {len(brazil_covid)} records")
brazil_covid.head()

Loading COVID-19 data for Brazil...


RuntimeError: Failed to download data from https://raw.githubusercontent.com/globaldothealth/list/main/data/cases.csv: HTTP Error 404: Not Found

## 4. Analyze Case Demographics

In [5]:
# Age distribution
if "age" in brazil_covid.columns:
    plt.figure(figsize=(10, 6))
    age_counts = brazil_covid["age"].value_counts().head(20)
    age_counts.plot(kind="bar")
    plt.title("COVID-19 Cases by Age - Brazil 2022")
    plt.xlabel("Age")
    plt.ylabel("Number of Cases")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

NameError: name 'brazil_covid' is not defined

In [ ]:
# Sex distribution
if "sex" in brazil_covid.columns:
    plt.figure(figsize=(8, 6))
    sex_counts = brazil_covid["sex"].value_counts()
    sex_counts.plot(kind="pie", autopct="%1.1f%%")
    plt.title("COVID-19 Cases by Sex - Brazil 2022")
    plt.ylabel("")
    plt.show()

## 5. Temporal Analysis

In [ ]:
# Cases over time
if "date_confirmation" in brazil_covid.columns:
    brazil_covid["date_confirmation"] = pd.to_datetime(brazil_covid["date_confirmation"], errors="coerce")
    daily_cases = brazil_covid.groupby(brazil_covid["date_confirmation"].dt.date).size()
    
    plt.figure(figsize=(12, 6))
    daily_cases.plot()
    plt.title("Daily COVID-19 Cases - Brazil 2022")
    plt.xlabel("Date")
    plt.ylabel("Number of Cases")
    plt.tight_layout()
    plt.show()

## 6. Compare Multiple Outbreaks

In [ ]:
# Compare COVID-19 and Monkeypox
comparison = gh.compare_outbreaks(["COVID-19", "Monkeypox"])
comparison

In [ ]:
# Visualize comparison
if not comparison.empty and "total_cases" in comparison.columns:
    plt.figure(figsize=(10, 6))
    comparison.plot(x="disease", y="total_cases", kind="bar", legend=False)
    plt.title("Outbreak Comparison - Total Cases")
    plt.xlabel("Disease")
    plt.ylabel("Total Cases")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 7. Geocoded Data

In [ ]:
# Get geocoded cases
geocoded = gh.get_geocoded_cases("COVID-19", country="Brazil")

# Check for coordinates
coord_cols = [col for col in geocoded.columns if any(x in col.lower() for x in ["lat", "lon", "coord"])]
print(f"Coordinate columns: {coord_cols}")

if coord_cols:
    # Simple scatter plot (if coordinates available)
    lat_col = [c for c in coord_cols if "lat" in c.lower()][0]
    lon_col = [c for c in coord_cols if "lon" in c.lower()][0]
    
    plt.figure(figsize=(10, 8))
    plt.scatter(geocoded[lon_col], geocoded[lat_col], alpha=0.5, s=10)
    plt.title("Geocoded COVID-19 Cases - Brazil")
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.show()

## Summary

This notebook demonstrated how to:
1. List available Global.health datasets
2. Retrieve outbreak metadata
3. Load case line-list data with filters
4. Analyze demographics and temporal trends
5. Compare multiple outbreaks
6. Work with geocoded data

For more information, visit: https://global.health/